# Multivariable Optimization

Companion notebook for the [Multivariable Optimization](https://ml-viz.vercel.app/courses/calculus-for-ml/03-multivariable-optimization) lesson.

We'll explore loss landscapes, classify critical points via the Hessian, and compare gradient descent learning rates.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#30344a', 'text.color': '#e2e8f0',
    'axes.labelcolor': '#e2e8f0', 'xtick.color': '#94a3b8', 'ytick.color': '#94a3b8',
})

## Visualizing different critical points

In [ ]:
x = np.linspace(-2, 2, 100)
y = np.linspace(-2, 2, 100)
X, Y = np.meshgrid(x, y)

functions = {
    'Local minimum\n(H positive definite)': X**2 + Y**2,
    'Saddle point\n(H indefinite)':          X**2 - Y**2,
    'Non-convex\n(multiple minima)':         np.sin(3*X) * np.cos(3*Y),
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5), subplot_kw={'projection': '3d'})

for ax, (title, Z) in zip(axes, functions.items()):
    surf = ax.plot_surface(X, Y, Z, cmap='twilight', alpha=0.85,
                           linewidth=0, antialiased=True)
    ax.set_title(title, pad=10, fontsize=10)
    ax.set_xlabel('w₁'); ax.set_ylabel('w₂'); ax.set_zlabel('Loss')
    ax.tick_params(labelsize=7)

plt.suptitle('Loss Landscape Shapes', y=1.02, fontsize=13)
plt.tight_layout(); plt.show()

## Hessian eigenvalues classify critical points

In [ ]:
def classify_critical_point(H):
    eigs = np.linalg.eigvalsh(H)
    if np.all(eigs > 0):  return 'Local minimum (all λ > 0)'
    if np.all(eigs < 0):  return 'Local maximum (all λ < 0)'
    if np.all(eigs == 0): return 'Degenerate'
    return 'Saddle point (mixed λ signs)'

hessians = {
    'f=x²+y²  at (0,0)': np.array([[2., 0.], [0., 2.]]),
    'f=x²-y²  at (0,0)': np.array([[2., 0.], [0.,-2.]]),
    'f=-x²-y² at (0,0)': np.array([[-2.,0.], [0.,-2.]]),
}

for name, H in hessians.items():
    eigs = np.linalg.eigvalsh(H)
    print(f'{name}')
    print(f'  Eigenvalues: {eigs}')
    print(f'  Classification: {classify_critical_point(H)}\n')

## Effect of learning rate on convergence

In [ ]:
# Quadratic with curvature L = 4 in x-direction, L = 1 in y-direction
def loss(w): return 2*w[0]**2 + 0.5*w[1]**2
def grad(w): return np.array([4*w[0], w[1]])

learning_rates = [0.05, 0.2, 0.49, 0.51]  # last two: stable vs unstable
w0 = np.array([2.0, 2.0])

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

xx, yy = np.meshgrid(np.linspace(-2.5, 2.5, 200), np.linspace(-2.5, 2.5, 200))
Z = 2*xx**2 + 0.5*yy**2

for ax, lr in zip(axes, learning_rates):
    path = [w0.copy()]
    w = w0.copy()
    for _ in range(40):
        w = w - lr * grad(w)
        path.append(w.copy())
        if np.any(np.abs(w) > 100):
            break
    path = np.array(path)

    ax.contourf(xx, yy, Z, levels=15, cmap='twilight', alpha=0.6)
    ax.contour(xx, yy, Z, levels=15, colors='white', alpha=0.2, linewidths=0.5)
    ax.plot(path[:, 0], path[:, 1], 'o-', color='#f97316', ms=4, lw=1.5)
    ax.scatter(*path[0], color='#2dd4bf', s=80, zorder=5, label='Start')
    ax.scatter(0, 0, marker='*', color='#f59e0b', s=150, zorder=5, label='Min')
    ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5)
    ax.set_aspect('equal'); ax.grid(False)
    final_loss = loss(path[-1])
    status = 'DIVERGED' if final_loss > 100 else f'Loss={final_loss:.3f}'
    ax.set_title(f'η = {lr}\n{status}', fontsize=10)

plt.suptitle('Effect of Learning Rate (optimal η = 1/L = 0.25)', y=1.04, fontsize=12)
plt.tight_layout(); plt.show()